# MediTrust — 02 Baselines: ResNet-18 vs DeiT-Tiny (Phase 2, E1)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/An4ss3/MediTrust/blob/main/notebooks/02_baselines.ipynb)

**Runtime: GPU (Runtime → Change runtime type → T4 GPU).** All logic lives in `src/`; this notebook only calls it.

| `MODE` | What it does | Approx. cost | Status |
|---|---|---|---|
| `"PILOT"` (default) | Architecture checks, ONNX check, 2-epoch run per model at lr 0.01 with a deliberate interruption and resume, resource measurements, budget estimate | ~4 epochs in total (~15–30 min on a T4, to be measured) | authorised |
| `"LR_SEARCH"` | Remaining runs of the learning-rate search (3 lr × 2 epochs × 2 models, protocol §4.1); the pilot runs are reused | ~8 more epochs | **only after the pilot is reviewed** |
| `"FULL_E1"` | 6 runs E1 (2 models × 3 seeds × 10 epochs) with the frozen learning rates | ~60 epochs | **only after approval**, needs `CONFIRM_FULL_E1 = True` |

The **test split is never used** in this notebook: it is removed from memory right after loading.
Checkpoints and JSON results go to Google Drive (`MyDrive/MediTrust/`), so an interrupted run resumes when the notebook is re-run.

## 0. Configuration

In [ ]:
MODE = "PILOT"            # "PILOT" | "LR_SEARCH" | "FULL_E1"
CONFIRM_FULL_E1 = False   # must be set to True explicitly for FULL_E1
FROZEN_LR = {"R18": None, "DEIT": None}  # filled in only after the LR search is reviewed and frozen
PILOT_LR = 0.01           # middle of the protocol grid; the pilot run is also the first LR-search run
LR_GRID = (0.003, 0.01, 0.03)
USE_DRIVE = True          # store checkpoints/results on Google Drive (recommended)

import os
SMOKE = os.environ.get("MEDITRUST_SMOKE") == "1"  # local CPU code check only: tiny data, 2 steps per epoch
assert MODE in ("PILOT", "LR_SEARCH", "FULL_E1"), MODE
print("MODE =", MODE, "| SMOKE =", SMOKE)

## 1. Environment and GPU

In [ ]:
import sys, platform, subprocess, time, json
IN_COLAB = "google.colab" in sys.modules
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Colab:", IN_COLAB, "| Python", platform.python_version(), "| torch", torch.__version__, "| CUDA", torch.version.cuda)
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory / 2**30:.1f} GiB")
    subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"])
elif not SMOKE:
    raise RuntimeError("No GPU: select Runtime > Change runtime type > T4 GPU")

## 2. Source code from GitHub and dependencies

In [ ]:
REPO_URL = "https://github.com/An4ss3/MediTrust.git"
if IN_COLAB:
    REPO_DIR = "/content/MediTrust"
    if os.path.isdir(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
else:
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()

import numpy as np, timm
from src import data as D, models as Mo, evaluate as Ev, train as Tr
print("commit", COMMIT, "| timm", timm.__version__, "| numpy", np.__version__)

## 3. Storage (Google Drive)

In [ ]:
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORE = "/content/drive/MyDrive/MediTrust"
else:
    STORE = os.path.join(REPO_DIR, "runs")  # ignored by git
CKPT_ROOT = os.path.join(STORE, "checkpoints")
RESULTS = os.path.join(STORE, "results")
for d in (CKPT_ROOT, RESULTS, os.path.join(STORE, "data")):
    os.makedirs(d, exist_ok=True)
print("checkpoints:", CKPT_ROOT, "\nresults:", RESULTS)

## 4. PathMNIST+ 64×64 (train and validation only)

In [ ]:
DATA_DIR = os.path.join(REPO_DIR, "data"); os.makedirs(DATA_DIR, exist_ok=True)
local, cached = os.path.join(DATA_DIR, D.FILENAME), os.path.join(STORE, "data", D.FILENAME)
if not os.path.exists(local) and os.path.exists(cached):
    subprocess.run(["cp", cached, local], check=True)
D.download(DATA_DIR)
if IN_COLAB and USE_DRIVE and not os.path.exists(cached):
    subprocess.run(["cp", local, cached], check=True)
data = D.load_pathmnist(DATA_DIR, check_md5=True)
train, val = data["train"], data["val"]
del data  # the test split is not kept in memory: no test evaluation in Phase 2 before the freeze
if SMOKE:
    train, val = (train[0][:256], train[1][:256]), (val[0][:180], val[1][:180])
print("train", train[0].shape, "| val", val[0].shape, "| MD5 OK")

## 5. Architecture checks (pretrained weights, before any training)

Forward shape, gradient flow to the 64×64 input, parameter count, explicit attention recomputation (DeiT),
ONNX export with ONNX Runtime parity on synthetic tensors, and the full pytest suite.

In [ ]:
import tempfile, onnxruntime as ort
checks = {}
x = torch.rand(4, 3, 64, 64)
for name in Mo.BACKBONES:
    m = Mo.build_model(name, pretrained=True).eval()
    xg = x.clone().requires_grad_(True)
    out = m(xg); out.logsumexp(1).sum().backward()
    c = {"backbone": Mo.BACKBONES[name], "params": Mo.count_parameters(m), "out_shape": list(out.shape),
         "input_grad_ok": bool(torch.isfinite(xg.grad).all() and xg.grad.abs().sum() > 0)}
    if name == "DEIT":
        tok = torch.randn(2, 197, 192)
        with torch.no_grad():
            o, w = Mo.manual_attention(Mo.attention_modules(m)[0], tok)
            c["attention_recompute_max_err"] = float((o - Mo.attention_modules(m)[0](tok)).abs().max())
    path = os.path.join(tempfile.gettempdir(), f"{name}.onnx")
    Mo.export_onnx(m, path)
    with torch.no_grad():
        ref = m(x).numpy()
    onnx_out = ort.InferenceSession(path, providers=["CPUExecutionProvider"]).run(None, {"image": x.numpy()})[0]
    c.update(onnx_mb=os.path.getsize(path) / 2**20, onnx_max_abs_diff=float(abs(onnx_out - ref).max()),
             onnx_argmax_equal=bool((onnx_out.argmax(1) == ref.argmax(1)).all()))
    checks[name] = c
    print(name, c)
assert all(c["input_grad_ok"] and c["onnx_argmax_equal"] and c["onnx_max_abs_diff"] < 1e-3 for c in checks.values())

In [ ]:
env = dict(os.environ, MEDITRUST_DATA=DATA_DIR)
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests"],
                   capture_output=True, text=True, env=env)
print(r.stdout[-1500:], r.stderr[-800:])
assert r.returncode == 0, "tests failed"

## 6. Pilot (MODE = "PILOT")

For each model: a 2-epoch run at lr = 0.01 (protocol configuration, seed 0). The run is **stopped after
epoch 1 on purpose** and then resumed from its checkpoint, which tests the resume path on the GPU.
This run is also the first run of the learning-rate search, so it is not wasted.
The validation metrics shown are not used for any decision at this stage.

In [ ]:
def lr_run_id(name, lr, seed=0):
    return f"LR-{name}-lr{lr}-s0" if seed == 0 else f"LR-{name}-lr{lr}-s{seed}"

STEPS = 2 if SMOKE else None
META = {"commit": COMMIT, "dataset_md5": D.MD5, "phase": 2, "experiment": "pilot / LR search"}
pilot = {}
if MODE == "PILOT":
    for name in Mo.BACKBONES:
        cfg = {"model": name, "lr": PILOT_LR, "epochs": 2, "seed": 0, "pretrained": not SMOKE}
        rid = lr_run_id(name, PILOT_LR)
        first = Tr.fit(cfg, train, val, rid, CKPT_ROOT, RESULTS, DEVICE, META,
                       stop_after_epoch=1, max_steps_per_epoch=STEPS)
        print(rid, "->", first["status"], "after epoch", first["epochs_completed"])
        pilot[name] = Tr.fit(cfg, train, val, rid, CKPT_ROOT, RESULTS, DEVICE, META, max_steps_per_epoch=STEPS)
        print(rid, "->", pilot[name]["status"], "| resumed from epoch", pilot[name]["resumed_from_epoch"])

## 7. Measured resources

In [ ]:
measured = {}
if MODE == "PILOT":
    print(f"{'model':5} {'epoch s':>8} {'img/s':>7} {'val s':>6} {'val img/s':>9} {'peak GPU MiB':>12} {'ckpt MiB':>8}  val acc / macro-F1 (epoch 2, not a decision)")
    for name, s in pilot.items():
        h = s["history"][-1]  # epoch 2: after warm-up of kernels
        measured[name] = {"epoch_seconds": h["train_seconds"], "train_images_per_second": h["train_images_per_second"],
                          "val_seconds": h["val_seconds"], "val_images_per_second": len(val[1]) / h["val_seconds"],
                          "peak_gpu_mb": s["peak_gpu_mb"], "checkpoint_mb": s["checkpoint_mb"],
                          "epoch1_seconds": s["history"][0]["train_seconds"]}
        m = measured[name]
        print(f"{name:5} {m['epoch_seconds']:8.1f} {m['train_images_per_second']:7.0f} {m['val_seconds']:6.1f} "
              f"{m['val_images_per_second']:9.0f} {m['peak_gpu_mb'] or 0:12.0f} {m['checkpoint_mb']:8.1f}  "
              f"{h['val']['accuracy']:.3f} / {h['val']['macro_f1']:.3f}")

## 8. Budget estimate for the full protocol (from the measurements)

One unit U = one training epoch on the 89,996 training images. Attack passes (FGSM/PGD) are counted as one
forward+backward each. The fallback level follows protocol §9.1 (N0 ≤ 15 h, N1 ≤ 25 h, N2 ≤ 40 h, with a 1.3 margin).
The level is **proposed** here; it is frozen only after review. The pipeline model M is not chosen yet, so both cases are shown.

In [ ]:
if MODE == "PILOT":
    # Extrapolated from measured throughput (equal to the measured epoch time for real runs).
    minutes_per_u = {n: D.N_SAMPLES["train"] / m["train_images_per_second"] / 60 for n, m in measured.items()}
    val_minutes = {n: D.N_SAMPLES["val"] / m["val_images_per_second"] / 60 for n, m in measured.items()}
    budget = Tr.estimate_budget(minutes_per_u, val_minutes)
    print("minutes per U:", {k: round(v, 2) for k, v in minutes_per_u.items()},
          "| minutes per validation pass:", {k: round(v, 2) for k, v in val_minutes.items()})
    for scenario, b in budget.items():
        print(f"\n{scenario}: proposed level {b['suggested_level']}")
        for lv, v in b["levels"].items():
            print(f"  {lv}: {v['units']:6.1f} U | {v['hours_raw']:5.1f} h raw | {v['hours_with_margin']:5.1f} h with margin")
        print(f"  contingency (strengthened PGD on alert): +{b['contingency_hours']:.1f} h")
    print("\nFree Colab sessions of ~2.5 h needed at N0 (worst case):",
          round(max(b["levels"]["N0"]["hours_with_margin"] for b in budget.values()) / 2.5, 1))

## 9. Save the pilot summary

In [ ]:
if MODE == "PILOT":
    summary = {"run_id": "P2-pilot", "date": time.strftime("%Y-%m-%d %H:%M:%S"), "commit": COMMIT, "smoke": SMOKE,
               "env": Tr.env_info(DEVICE), "checks": checks, "measured": measured,
               "minutes_per_u": minutes_per_u, "budget": budget,
               "runs": {n: s["run_id"] for n, s in pilot.items()}}
    for path in (os.path.join(RESULTS, "P2-pilot.json"), "results/P2-pilot.json"):
        with open(path, "w") as f:
            json.dump(summary, f, indent=1)
    print("saved results/P2-pilot.json (and a copy on Drive). Download it and send it for review.")

## 10. Learning-rate search (MODE = "LR_SEARCH", only after the pilot review)

Grid {0.003, 0.01, 0.03} × 2 epochs × 2 models, seed 0. Completed runs (including the pilot) are skipped.
Rule (protocol §4.1): highest validation macro-F1; if two values are within 0.1 point, the lower validation NLL.
The printed choice is a **proposal**: it is frozen in `docs/PROJECT_STATUS.md` before `FULL_E1`.

In [ ]:
if MODE == "LR_SEARCH":
    table = {}
    for name in Mo.BACKBONES:
        for lr in LR_GRID:
            cfg = {"model": name, "lr": lr, "epochs": 2, "seed": 0, "pretrained": not SMOKE}
            s = Tr.fit(cfg, train, val, lr_run_id(name, lr), CKPT_ROOT, RESULTS, DEVICE, META, max_steps_per_epoch=STEPS)
            v = s["history"][-1]["val"]
            table[(name, lr)] = v
            print(f"{name:5} lr={lr:<6} val macro-F1 {v['macro_f1']:.4f} | acc {v['accuracy']:.4f} | NLL {v['nll']:.4f}")
    for name in Mo.BACKBONES:
        rows = sorted(((v["macro_f1"], -v["nll"], lr) for (n, lr), v in table.items() if n == name), reverse=True)
        best_f1 = rows[0][0]
        tied = [r for r in rows if best_f1 - r[0] <= 0.001]
        print(f"proposed lr for {name}: {max(tied, key=lambda r: r[1])[2]}")

## 11. Full E1 (MODE = "FULL_E1", only after approval)

In [ ]:
if MODE == "FULL_E1":
    assert CONFIRM_FULL_E1, "Set CONFIRM_FULL_E1 = True only after the pilot and the LR search are approved"
    assert all(FROZEN_LR.values()), "Fill FROZEN_LR with the frozen learning rates"
    for name in Mo.BACKBONES:
        for seed in (0, 1, 2):
            cfg = {"model": name, "lr": FROZEN_LR[name], "epochs": 10, "seed": seed}
            s = Tr.fit(cfg, train, val, f"E1-{name}-s{seed}", CKPT_ROOT, RESULTS, DEVICE,
                       {**META, "experiment": "E1"})
            print(s["run_id"], s["status"], {k: round(s["val_final"][k], 4) for k in ("accuracy", "macro_f1", "nll")})
    print("E1 training done. No test evaluation here: freeze D3 first (protocol §7).")